<a href="https://colab.research.google.com/github/nishnarudkar/Interpretable-Machine-Learning-System-for-Parkinson-s-Disease-Detection-from-Speech-Biomarkers/blob/main/notebooks/Parkinsons_Detection_MLOPS_Project_SMOTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, learning_curve, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Parkinson's Dataset for Mlops Project/pd_speech_features.csv", header=1)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df = df.drop("id", axis=1)

In [ ]:
df.shape

In [ ]:
df["class"].value_counts()

In [ ]:
X = df.drop("class", axis=1)
y = df["class"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

## Feature Selection

SMOTE is applied first so the RF selector learns minority-class features,
then selection is applied to the original (non-SMOTE) train split to avoid leakage.

In [ ]:
# Use SMOTE only to help selector learn minority features
smote_fs = SMOTE(random_state=42)
X_train_fs, y_train_fs = smote_fs.fit_resample(X_train, y_train)

rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
selector = SelectFromModel(rf_selector, max_features=100)
selector.fit(X_train_fs, y_train_fs)

X_train_sel = selector.transform(X_train)
X_test_sel  = selector.transform(X_test)
print(f"Selected features: {X_train_sel.shape[1]}")

In [ ]:
selected_indices = selector.get_support(indices=True)
selected_features = X.columns[selected_indices]

## Scaling (select → scale order)

In [ ]:
# Scaler kept for the production XGBoost serving pipeline
# (saved as artifact and used by api/main.py at inference time)
scaler = StandardScaler()
X_train_sel_scaled = scaler.fit_transform(X_train_sel)
X_test_sel_scaled  = scaler.transform(X_test_sel)


## Helper: compute all metrics (macro-averaged)

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro"),
        "recall":    recall_score(y_true, y_pred, average="macro"),
        "macro_f1":  f1_score(y_true, y_pred, average="macro"),
        "roc_auc":   roc_auc_score(y_true, y_prob),
    }

model_results = []
best_f1    = 0
best_model = None

### Models


#### Logistic Regression

In [ ]:
print("--- Tuning Logistic Regression ---")
lr_pipeline = ImbPipeline([
    ("smote",  SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("lr",     LogisticRegression(random_state=42, max_iter=5000)),
])
lr_param_dist = {
    "lr__C":      uniform(0.01, 10),
    "lr__solver": ["lbfgs", "saga"],
}
lr_search = RandomizedSearchCV(
    lr_pipeline, lr_param_dist, n_iter=20,
    scoring="f1_macro", cv=StratifiedKFold(5, shuffle=True, random_state=42),
    n_jobs=-1, random_state=42, verbose=0
)
lr_search.fit(X_train_sel, y_train)  # unscaled — pipeline scales inside each fold
best_lr_pipeline = lr_search.best_estimator_

y_pred_lr = best_lr_pipeline.predict(X_test_sel)
y_prob_lr = best_lr_pipeline.predict_proba(X_test_sel)[:, 1]
lr_metrics = compute_metrics(y_test, y_pred_lr, y_prob_lr)

model_results.append({"Model": "Logistic Regression", **lr_metrics})
print(f"\nLogisticRegression: {lr_metrics}")
print(classification_report(y_test, y_pred_lr))

if lr_metrics["macro_f1"] > best_f1:
    best_f1    = lr_metrics["macro_f1"]
    best_model = best_lr_pipeline.named_steps["lr"]


#### Random Forest

In [ ]:
print("--- Tuning Random Forest ---")
rf_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("rf",    RandomForestClassifier(random_state=42)),
])
rf_param_dist = {
    "rf__n_estimators":      randint(100, 400),
    "rf__max_depth":         [None, 10, 20, 30],
    "rf__max_features":      ["sqrt", "log2"],
    "rf__min_samples_split": randint(2, 10),
}
rf_search = RandomizedSearchCV(
    rf_pipeline, rf_param_dist, n_iter=20,
    scoring="f1_macro", cv=StratifiedKFold(5, shuffle=True, random_state=42),
    n_jobs=-1, random_state=42, verbose=0
)
rf_search.fit(X_train_sel, y_train)
best_rf_pipeline = rf_search.best_estimator_

y_pred_rf = best_rf_pipeline.predict(X_test_sel)
y_prob_rf = best_rf_pipeline.predict_proba(X_test_sel)[:, 1]
rf_metrics = compute_metrics(y_test, y_pred_rf, y_prob_rf)

model_results.append({"Model": "Random Forest", **rf_metrics})
print(f"\nRandomForest: {rf_metrics}")
print(classification_report(y_test, y_pred_rf))

if rf_metrics["macro_f1"] > best_f1:
    best_f1    = rf_metrics["macro_f1"]
    best_model = best_rf_pipeline.named_steps["rf"]

#### Support Vector Machine (SVM)

In [ ]:
print("--- Tuning SVM ---")
svm_pipeline = ImbPipeline([
    ("smote",  SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("svm",    SVC(probability=True, random_state=42)),
])
svm_param_dist = {
    "svm__C":      uniform(0.1, 10),
    "svm__gamma":  ["scale", "auto"],
    "svm__kernel": ["rbf", "poly"],
}
svm_search = RandomizedSearchCV(
    svm_pipeline, svm_param_dist, n_iter=15,
    scoring="f1_macro", cv=StratifiedKFold(5, shuffle=True, random_state=42),
    n_jobs=-1, random_state=42, verbose=0
)
svm_search.fit(X_train_sel, y_train)  # unscaled — pipeline scales inside each fold
best_svm_pipeline = svm_search.best_estimator_

y_pred_svm = best_svm_pipeline.predict(X_test_sel)
y_prob_svm = best_svm_pipeline.predict_proba(X_test_sel)[:, 1]
svm_metrics = compute_metrics(y_test, y_pred_svm, y_prob_svm)

model_results.append({"Model": "SVM", **svm_metrics})
print(f"\nSVM: {svm_metrics}")
print(classification_report(y_test, y_pred_svm))

if svm_metrics["macro_f1"] > best_f1:
    best_f1    = svm_metrics["macro_f1"]
    best_model = best_svm_pipeline.named_steps["svm"]


#### KNN

In [ ]:
print("--- Tuning KNN ---")
knn_pipeline = ImbPipeline([
    ("smote",  SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsClassifier()),
])
knn_param_dist = {
    "knn__n_neighbors": randint(3, 20),
    "knn__weights":     ["uniform", "distance"],
    "knn__metric":      ["euclidean", "manhattan"],
}
knn_search = RandomizedSearchCV(
    knn_pipeline, knn_param_dist, n_iter=15,
    scoring="f1_macro", cv=StratifiedKFold(5, shuffle=True, random_state=42),
    n_jobs=-1, random_state=42, verbose=0
)
knn_search.fit(X_train_sel, y_train)  # unscaled — pipeline scales inside each fold
best_knn_pipeline = knn_search.best_estimator_

y_pred_knn = best_knn_pipeline.predict(X_test_sel)
y_prob_knn = best_knn_pipeline.predict_proba(X_test_sel)[:, 1]
knn_metrics = compute_metrics(y_test, y_pred_knn, y_prob_knn)

model_results.append({"Model": "KNN", **knn_metrics})
print(f"\nKNN: {knn_metrics}")
print(classification_report(y_test, y_pred_knn))

if knn_metrics["macro_f1"] > best_f1:
    best_f1    = knn_metrics["macro_f1"]
    best_model = best_knn_pipeline.named_steps["knn"]


#### Decision Tree

In [ ]:
print("--- Tuning Decision Tree ---")
dt_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("dt",    DecisionTreeClassifier(random_state=42)),
])
dt_param_dist = {
    "dt__max_depth":         [None, 5, 10, 15, 20],
    "dt__min_samples_split": randint(2, 20),
    "dt__min_samples_leaf":  randint(1, 10),
    "dt__criterion":         ["gini", "entropy"],
}
dt_search = RandomizedSearchCV(
    dt_pipeline, dt_param_dist, n_iter=20,
    scoring="f1_macro", cv=StratifiedKFold(5, shuffle=True, random_state=42),
    n_jobs=-1, random_state=42, verbose=0
)
dt_search.fit(X_train_sel, y_train)
best_dt_pipeline = dt_search.best_estimator_

y_pred_dt = best_dt_pipeline.predict(X_test_sel)
y_prob_dt = best_dt_pipeline.predict_proba(X_test_sel)[:, 1]
dt_metrics = compute_metrics(y_test, y_pred_dt, y_prob_dt)

model_results.append({"Model": "Decision Tree", **dt_metrics})
print(f"\nDecisionTree: {dt_metrics}")
print(classification_report(y_test, y_pred_dt))

if dt_metrics["macro_f1"] > best_f1:
    best_f1    = dt_metrics["macro_f1"]
    best_model = best_dt_pipeline.named_steps["dt"]

#### XGBoost — RandomizedSearchCV with leakage-free ImbPipeline

In [ ]:
print("--- Tuning XGBoost (RandomizedSearchCV) ---")

xgb_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("xgb",   XGBClassifier(learning_rate=0.05, eval_metric="logloss", random_state=42)),
])

xgb_param_dist = {
    "xgb__max_depth":        randint(3, 8),
    "xgb__min_child_weight": randint(1, 6),
    "xgb__gamma":            uniform(0, 0.5),
    "xgb__subsample":        uniform(0.7, 0.3),
    "xgb__colsample_bytree": uniform(0.7, 0.3),
    "xgb__n_estimators":     randint(100, 400),
}

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    xgb_param_dist,
    n_iter=30,
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

xgb_search.fit(X_train_sel, y_train)

best_tuned_pipeline = xgb_search.best_estimator_
best_tuned_xgb      = best_tuned_pipeline.named_steps["xgb"]

y_pred_tuned = best_tuned_pipeline.predict(X_test_sel)
y_prob_tuned = best_tuned_pipeline.predict_proba(X_test_sel)[:, 1]
xgb_metrics  = compute_metrics(y_test, y_pred_tuned, y_prob_tuned)

model_results.append({"Model": "XGBoost", **xgb_metrics})
print(f"\nBest XGBoost params: {xgb_search.best_params_}")
for k, v in xgb_metrics.items():
    print(f"  {k}: {v:.4f}")
print(classification_report(y_test, y_pred_tuned))

if xgb_metrics["macro_f1"] > best_f1:
    best_f1    = xgb_metrics["macro_f1"]
    best_model = best_tuned_xgb

### Model Leaderboard

In [ ]:
df_leaderboard = pd.DataFrame(model_results).sort_values(by="macro_f1", ascending=False)
print("\n🏆 Model Leaderboard:")
print(df_leaderboard.to_string(index=False))

### Production Model Selection

XGBoost is always selected as the production model regardless of leaderboard rank.
Rationale: this is a medical application where interpretability is critical.
XGBoost supports fast, exact SHAP (TreeExplainer) which provides meaningful
biomarker explanations.

In [ ]:
# XGBoost is always used as the production model regardless of leaderboard rank.
production_model = best_tuned_xgb
print(f"Leaderboard winner:  {type(best_model).__name__} (macro F1: {best_f1:.4f})")
print(f"Production model: XGBoost (macro F1: {xgb_metrics['macro_f1']:.4f})")
print("Reason: XGBoost selected for interpretability (SHAP TreeExplainer) in medical context.")

### Confusion Matrix & ROC Curve

In [ ]:
plt.figure()
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_tuned)
plt.title("Confusion Matrix (XGBoost)")
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob_tuned)
plt.title("ROC Curve (XGBoost)")
plt.savefig("roc_curve.png", dpi=300)
plt.show()

### Learning Curve — Full ImbPipeline (no leakage)

The full preprocessing pipeline (SMOTE → SelectFromModel → StandardScaler) is
refitted inside each CV fold so there is no data leakage.

In [ ]:
lc_pipeline = ImbPipeline([
    ('smote',    SMOTE(random_state=42)),
    ('selector', SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42), max_features=100)),
    ('scaler',   StandardScaler()),
    ('model',    production_model)
])

train_sizes, train_scores, val_scores = learning_curve(
    lc_pipeline, X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1_macro",
    train_sizes=np.linspace(0.2, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, label="Train",      color="#6c63ff", linewidth=2)
ax.plot(train_sizes, val_mean,   label="Validation", color="#34d399", linewidth=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="#6c63ff")
ax.fill_between(train_sizes, val_mean - val_std,     val_mean + val_std,     alpha=0.15, color="#34d399")

gap = train_mean[-1] - val_mean[-1]
ax.annotate(
    f"Gap: {gap:.3f}",
    xy=(train_sizes[-1], (train_mean[-1] + val_mean[-1]) / 2),
    xytext=(-80, 0), textcoords="offset points",
    fontsize=9, color="#f87171",
    arrowprops=dict(arrowstyle="->", color="#f87171"),
)
ax.set_xlabel("Training set size")
ax.set_ylabel("Macro F1 Score")
ax.set_title("Learning Curve — Bias/Variance Analysis\n(preprocessing refitted per fold — no leakage)")
ax.legend()
ax.set_ylim(0, 1.05)
fig.tight_layout()
plt.savefig("learning_curve.png", dpi=150)
plt.show()

print(f"Train F1: {train_mean[-1]:.4f} | Val F1: {val_mean[-1]:.4f} | Gap: {gap:.4f}")

### SHAP Explainability

In [ ]:
def extract_shap_for_class1(raw):
    """Robustly extract a 2-D SHAP array (n_samples, n_features) for class 1,
    regardless of SHAP version or model type output format."""
    if isinstance(raw, list):
        arr = np.array(raw[1])
    else:
        arr = np.array(raw)
    if arr.ndim == 3:
        arr = arr[:, :, 1]   # (n_samples, n_features, n_classes) → class-1
    if arr.ndim == 1:
        arr = arr[np.newaxis, :]  # single sample → (1, n_features)
    return arr  # (n_samples, n_features)

In [ ]:
# Use scaled data for SHAP (same preprocessing as training)
X_sample = X_test_sel_scaled[:50]

print("Creating SHAP TreeExplainer for XGBoost...")
explainer   = shap.TreeExplainer(production_model)
shap_values = explainer.shap_values(X_sample)

shap_2d    = extract_shap_for_class1(shap_values)
importance = np.abs(shap_2d).mean(axis=0)

feature_importance = pd.DataFrame({
    "feature":    selected_features,
    "importance": importance,
}).sort_values("importance", ascending=False)

top_features = feature_importance.head(20)

plt.figure(figsize=(10, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.gca().invert_yaxis()
plt.xlabel("Mean |SHAP value|")
plt.title("Top 20 Speech Biomarkers for Parkinson Detection\n(XGBoost)")
plt.tight_layout()
plt.savefig("shap_bar.png", bbox_inches="tight")
plt.show()

print("\nTop 10 features by SHAP importance:")
print(feature_importance.head(10).to_string(index=False))

In [ ]:
# SHAP summary plot
shap.summary_plot(shap_2d, X_sample, feature_names=list(selected_features), show=True)